# Capa Gold — Preguntas 6 a 8 (las "inteligentes": cruzar inversión con embudo)

**Qué hace esta notebook:** las 3 preguntas que ninguna capa podía responder sola — cruzar de dónde viene el dinero (advertiser/campaña/país) con qué devuelve (conversiones).

| # | Pregunta | Tabla gold |
|---|---|---|
| 6 | ¿Qué industrias recuperan más dinero por cada dólar invertido (ROAS)? | `q6_roas_por_industria` |
| 7 | ¿Qué campañas gastan plata sin generar ni una conversión? | `q7_campanas_sin_conversiones` |
| 8 | ¿Qué países convierten mejor respecto a las impresiones que reciben? | `q8_paises_mejor_conversion` |

**De dónde salen los datos:** la tabla puente `clicks_atribuidos` + `conversions_atribuidas` (notebook 05) ya resuelve la atribución y ahora también lleva el `pais` de la impresión atribuida. Esta notebook solo joinea contra ella — cero lógica de atribución duplicada.

**Decisión de negocio Q6:** ROAS = ingresos (`value_usd` de las conversiones) / gasto (costo de impresiones + clics). Si ROAS > 1, la industria recupera más de lo que gasta.

**Decisión de negocio Q7:** "presupuesto quemado" = campaña con gasto real > 0 y conversiones = 0. Con solo 256 conversiones para 2.000 campañas, la mayoría va a aparecer acá — el valor está en el RANKING por gasto (las peores primero), no en el conteo.

**Decisión de negocio Q8:** umbral anti-ruido de 500 impresiones mínimas por país (medido: hay 195 países con ~820 impresiones cada uno). Sin umbral, un país con 3 impresiones y 1 conversión "ganaría" la tabla con 33%.

**Idempotencia:** igual que el resto del gold, `CREATE OR REPLACE TABLE` (recalcula desde silver/puente; re-ejecutar N veces no cambia nada).

In [0]:
%sql
-- PREGUNTA 6: ROAS por industria (Return On Ad Spend).
-- ROAS = ingresos / gasto. Mayor a 1 = se recupera más de lo invertido.
-- Joins:
--   gasto de impresiones: impressions -> ads -> campaigns -> advertisers (industria)
--   gasto de clics:       clicks_atribuidos -> campaigns -> advertisers
--   ingresos:             conversions_atribuidas -> campaigns -> advertisers (tabla puente)
-- Funciones de ventana: RANK() rankeando industrias por ROAS y SUM() OVER () para el ROAS
-- global de referencia (leer cada industria CONTRA el promedio, no como número suelto).

CREATE OR REPLACE TABLE dev.gold_ads.q6_roas_por_industria AS
WITH gasto_imp AS (
  SELECT a.industry AS industria, COALESCE(SUM(i.cost_usd), 0) AS gasto_impresiones_usd
  FROM dev.silver_ads.impressions i
  JOIN dev.silver_ads.ads a2    ON a2.id = i.ad_id
  JOIN dev.silver_ads.campaigns c  ON c.id  = a2.campaign_id
  JOIN dev.silver_ads.advertisers a ON a.id = c.advertiser_id
  GROUP BY a.industry
),
gasto_clicks AS (
  SELECT ad.industry AS industria, COALESCE(SUM(ca.click_cost_usd), 0) AS gasto_clics_usd
  FROM dev.gold_ads.clicks_atribuidos ca
  JOIN dev.silver_ads.campaigns c ON c.id = ca.campaign_id
  JOIN dev.silver_ads.advertisers ad ON ad.id = c.advertiser_id
  GROUP BY ad.industry
),
ingresos AS (
  SELECT ad.industry AS industria, SUM(v.value_usd) AS ingresos_usd, COUNT(*) AS conversiones
  FROM dev.gold_ads.conversions_atribuidas v
  JOIN dev.silver_ads.campaigns c ON c.id = v.campaign_id
  JOIN dev.silver_ads.advertisers ad ON ad.id = c.advertiser_id
  GROUP BY ad.industry
)
SELECT
  gi.industria,
  ROUND(gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0), 4) AS gasto_total_usd,
  COALESCE(ig.conversiones, 0) AS conversiones,
  ROUND(COALESCE(ig.ingresos_usd, 0), 2) AS ingresos_usd,
  ROUND(COALESCE(ig.ingresos_usd, 0)
        / NULLIF(gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0), 0), 4) AS roas,
  -- ROAS global de referencia: ingresos totales / gasto total (ventana sobre TODAS las industrias)
  ROUND(SUM(COALESCE(ig.ingresos_usd, 0)) OVER ()
        / SUM(gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0)) OVER (), 4) AS roas_global,
  -- Brecha: cuánto está la industria por encima (+) o debajo (-) del ROAS global
  ROUND(COALESCE(ig.ingresos_usd, 0)
        / NULLIF(gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0), 0)
        - SUM(COALESCE(ig.ingresos_usd, 0)) OVER ()
          / SUM(gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0)) OVER (), 4) AS brecha_vs_global,
  RANK() OVER (ORDER BY COALESCE(ig.ingresos_usd, 0)
               / (gi.gasto_impresiones_usd + COALESCE(gc.gasto_clics_usd, 0)) DESC) AS ranking_roas
FROM gasto_imp gi
LEFT JOIN gasto_clicks gc ON gc.industria = gi.industria
LEFT JOIN ingresos ig    ON ig.industria = gi.industria;

SELECT * FROM dev.gold_ads.q6_roas_por_industria ORDER BY ranking_roas;

industria,gasto_total_usd,conversiones,ingresos_usd,roas,roas_global,brecha_vs_global,ranking_roas
Retail,1196.297,73,1819.68,1.5211,0.8865,0.6346,1
FinTech,1498.907,54,1230.72,0.8211,0.8865,-0.0654,2
Gaming,1600.9459,63,1305.08,0.8152,0.8865,-0.0713,3
SaaS,1071.9642,33,777.87,0.7256,0.8865,-0.1608,4
Travel,1208.112,33,696.45,0.5765,0.8865,-0.31,5


In [0]:
%sql
-- GRÁFICO: ROAS por industria vs el ROAS global (barras).
-- Aporta valor porque la lectura es COMPARATIVA: Retail sobresale del promedio y Travel
-- queda clara como la peor. La brecha numérica se ve como altura de barra.
SELECT industria, roas, brecha_vs_global
FROM dev.gold_ads.q6_roas_por_industria
ORDER BY ranking_roas;

industria,roas,brecha_vs_global
Retail,1.5211,0.6346
FinTech,0.8211,-0.0654
Gaming,0.8152,-0.0713
SaaS,0.7256,-0.1608
Travel,0.5765,-0.31


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- PREGUNTA 7: campañas que gastan plata SIN generar ni una conversión.
-- Regla de negocio: gasto_real > 0 AND conversiones = 0.
-- Joins: impressions -> ads -> campaigns (gasto de impresiones);
--        clicks_atribuidos (clics y su costo por campaña);
--        conversions_atribuidas (conversiones por campaña).
-- Función de ventana: RANK() por gasto real (las peores gastadoras sin retorno primero).
-- Ojo con la lectura: con 256 conversiones para 2.000 campañas, la mayoría no convierte
-- (es esperado). El valor de la tabla es el RANKING: a igual cero conversiones, cuáles
-- queman más plata.

CREATE OR REPLACE TABLE dev.gold_ads.q7_campanas_sin_conversiones AS
WITH gasto AS (
  SELECT a.campaign_id,
         COUNT(DISTINCT i.event_key) AS impresiones,
         COALESCE(SUM(i.cost_usd), 0) AS gasto_impresiones_usd
  FROM dev.silver_ads.impressions i
  JOIN dev.silver_ads.ads a ON a.id = i.ad_id
  GROUP BY a.campaign_id
),
clics AS (
  SELECT campaign_id, COUNT(*) AS clics, COALESCE(SUM(click_cost_usd), 0) AS gasto_clics_usd
  FROM dev.gold_ads.clicks_atribuidos
  GROUP BY campaign_id
),
conv AS (
  SELECT campaign_id, COUNT(*) AS conversiones
  FROM dev.gold_ads.conversions_atribuidas
  GROUP BY campaign_id
)
SELECT
  c.id           AS campaign_id,
  c.campaign_name,
  c.channel      AS canal,
  ad.industry    AS industria,
  ad.advertiser_name,
  COALESCE(g.impresiones, 0) AS impresiones,
  COALESCE(k.clics, 0) AS clics,
  ROUND(COALESCE(g.gasto_impresiones_usd, 0) + COALESCE(k.gasto_clics_usd, 0), 4) AS gasto_real_usd,
  RANK() OVER (ORDER BY COALESCE(g.gasto_impresiones_usd, 0) + COALESCE(k.gasto_clics_usd, 0) DESC) AS ranking_gasto_sin_retorno
FROM dev.silver_ads.campaigns c
JOIN dev.silver_ads.advertisers ad ON ad.id = c.advertiser_id
LEFT JOIN gasto g ON g.campaign_id = c.id
LEFT JOIN clics k ON k.campaign_id = c.id
LEFT JOIN conv v ON v.campaign_id = c.id
WHERE COALESCE(v.conversiones, 0) = 0
  AND COALESCE(g.gasto_impresiones_usd, 0) + COALESCE(k.gasto_clics_usd, 0) > 0;

-- Vista rápida: las 10 campañas que más plata quemaron sin ninguna conversión
SELECT campaign_id, campaign_name, canal, industria, impresiones, clics, gasto_real_usd
FROM dev.gold_ads.q7_campanas_sin_conversiones
ORDER BY ranking_gasto_sin_retorno
LIMIT 10;

-- RESUMEN EJECUTIVO: ¿cuánta plata se quemó sin retorno y en qué canal se concentra?
-- (la tabla completa tiene ~1.900 filas; este es el headline que nadie quiere leer en una lista)
SELECT
  canal,
  COUNT(*) AS campanas_sin_conversion,
  ROUND(SUM(gasto_real_usd), 2) AS gasto_quemado_usd,
  ROUND(100.0 * SUM(gasto_real_usd) / SUM(SUM(gasto_real_usd)) OVER (), 1) AS pct_del_gasto_quemado
FROM dev.gold_ads.q7_campanas_sin_conversiones
GROUP BY canal
ORDER BY gasto_quemado_usd DESC;

canal,campanas_sin_conversion,gasto_quemado_usd,pct_del_gasto_quemado
Search,466,1550.66,25.5
Display,477,1515.54,24.9
Social,485,1509.57,24.8
Video,475,1508.69,24.8


In [0]:
%sql
-- GRÁFICO: las 10 campañas que más plata quemaron sin conversiones (barras).
-- Aporta valor porque el ranking se ve ordenado de un golpe, sin escanear 1.900 filas.
SELECT campaign_name, gasto_real_usd
FROM dev.gold_ads.q7_campanas_sin_conversiones
ORDER BY ranking_gasto_sin_retorno
LIMIT 10;

campaign_name,gasto_real_usd
Enable Global Web Services,15.2693
Streamline Value-Added E-Business,13.8615
Scale Impactful Paradigms,13.5452
Extend Intuitive Experiences,13.5235
Envisioneer World-Class E-Business,13.3822
Revolutionize Collaborative Bandwidth,13.2468
Reinvent Out-Of-The-Box E-Markets,12.9344
Streamline Visionary Technologies,12.7759
Brand Back-End Solutions,12.6204
Synergize Real-Time Applications,12.6162


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- PREGUNTA 8: países con mejor tasa de conversión respecto a las impresiones que reciben.
-- El país de un clic/conversión = el de su impresión atribuida (la tabla puente ya lo lleva).
-- UMBRAL anti-ruido: solo países con >= 500 impresiones (medido: 195 países, ~820 c/u).
-- Sin umbral, un país con 3 impresiones y 1 conversión "ganaría" con 33%.
-- Métricas: tasa_conversion_pct (conversiones / impresiones * 100) y ahora también la
-- ECONOMÍA por país: gasto total, ingresos y CPA (cuánto cuesta conseguir un cliente allí).
-- Pasa de "¿dónde convierten más?" a "¿dónde sale más barato conseguir un cliente?".
-- Funciones de ventana: RANK() por tasa de conversión y RANK() por CPA más barato.

CREATE OR REPLACE TABLE dev.gold_ads.q8_paises_mejor_conversion AS
WITH imp AS (
  SELECT country AS pais, COUNT(*) AS impresiones, COALESCE(SUM(cost_usd), 0) AS gasto_impresiones_usd
  FROM dev.silver_ads.impressions
  GROUP BY country
  HAVING COUNT(*) >= 500
),
clics AS (
  SELECT pais, COUNT(*) AS clics, COALESCE(SUM(click_cost_usd), 0) AS gasto_clics_usd
  FROM dev.gold_ads.clicks_atribuidos
  GROUP BY pais
),
cnv AS (
  SELECT pais, COUNT(*) AS conversiones, COALESCE(SUM(value_usd), 0) AS ingresos_usd
  FROM dev.gold_ads.conversions_atribuidas
  GROUP BY pais
)
SELECT
  imp.pais,
  imp.impresiones,
  COALESCE(cl.clics, 0) AS clics,
  COALESCE(cv.conversiones, 0) AS conversiones,
  ROUND(100.0 * COALESCE(cv.conversiones, 0) / imp.impresiones, 4) AS tasa_conversion_pct,
  ROUND(imp.gasto_impresiones_usd + COALESCE(cl.gasto_clics_usd, 0), 4) AS gasto_total_usd,
  ROUND(COALESCE(cv.ingresos_usd, 0), 2) AS ingresos_usd,
  -- CPA del país: NULL si no convirtió nadie (no se divide por 0)
  ROUND((imp.gasto_impresiones_usd + COALESCE(cl.gasto_clics_usd, 0))
        / NULLIF(COALESCE(cv.conversiones, 0), 0), 4) AS cpa_usd,
  RANK() OVER (ORDER BY 100.0 * COALESCE(cv.conversiones, 0) / imp.impresiones DESC) AS ranking_conversion,
  RANK() OVER (ORDER BY (imp.gasto_impresiones_usd + COALESCE(cl.gasto_clics_usd, 0))
               / NULLIF(COALESCE(cv.conversiones, 0), 0) ASC NULLS LAST) AS ranking_cpa_mas_barato
FROM imp
LEFT JOIN clics cl ON cl.pais = imp.pais
LEFT JOIN cnv cv  ON cv.pais = imp.pais;

-- Vista rápida: top 10 países por tasa de conversión
SELECT pais, impresiones, conversiones, tasa_conversion_pct, ingresos_usd
FROM dev.gold_ads.q8_paises_mejor_conversion
ORDER BY ranking_conversion
LIMIT 10;

-- ¿Y dónde sale MÁS BARATO conseguir un cliente? Top 10 países por CPA
SELECT pais, conversiones, gasto_total_usd, ingresos_usd, cpa_usd
FROM dev.gold_ads.q8_paises_mejor_conversion
WHERE conversiones > 0
ORDER BY ranking_cpa_mas_barato
LIMIT 10;

pais,conversiones,gasto_total_usd,ingresos_usd,cpa_usd
GR,4,14.7269,122.51,3.6817
CV,6,22.1628,73.4,3.6938
KG,6,27.3254,89.01,4.5542
LC,5,24.36,139.89,4.872
HN,8,38.986,191.93,4.8733
NR,6,29.3778,156.3,4.8963
ER,7,37.1417,139.01,5.306
LS,7,38.1153,149.19,5.445
CR,4,22.2666,133.13,5.5667
CN,7,39.9461,133.02,5.7066


In [0]:
%sql
-- GRÁFICO: top 10 países por tasa de conversión sobre impresiones (barras).
-- Aporta valor porque la pregunta es un RANKING: se ve qué países encabezan y qué tan parejas
-- son las tasas entre ellos (spoiler: casi uniforme, y eso también es un insight).
SELECT pais, tasa_conversion_pct
FROM dev.gold_ads.q8_paises_mejor_conversion
WHERE conversiones > 0
ORDER BY ranking_conversion
LIMIT 10;

pais,tasa_conversion_pct
HN,0.9852
LS,0.8728
TL,0.8674
CN,0.8444
ER,0.7919
NR,0.7772
CV,0.7673
UY,0.7444
CI,0.7398
AM,0.7273


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- VERIFICACIÓN: el gold 6-8 tiene que cerrar con el gold 1-5 y con silver.
--   gasto_q6 = gasto_q4      (mismo gasto total visto por industria y por canal)
--   conv_q6 = conv_silver = 256, ingresos_q6 = ingresos_silver (ninguna conversión se pierde)
--   violaciones_regla_q7 = 0 (ninguna campaña de la lista "sin conversiones" tiene en realidad conversiones)
--   imp_q8 = imp_silver y conv_q8 = 256 (el umbral no descartó ningún país: todos tienen >= 500)
--   gasto_q8 = gasto_q4 = gasto_q6 (la economía por país cierra con el total del período)
SELECT
  (SELECT ROUND(SUM(gasto_total_usd), 4) FROM dev.gold_ads.q6_roas_por_industria) AS gasto_q6,
  (SELECT ROUND(SUM(gasto_total_usd), 4) FROM dev.gold_ads.q4_cpa_por_canal)      AS gasto_q4,
  (SELECT SUM(conversiones) FROM dev.gold_ads.q6_roas_por_industria)  AS conv_q6,
  (SELECT COUNT(*) FROM dev.silver_ads.conversions)                    AS conv_silver,
  (SELECT ROUND(SUM(ingresos_usd), 2) FROM dev.gold_ads.q6_roas_por_industria) AS ingresos_q6,
  (SELECT ROUND(SUM(value_usd), 2) FROM dev.silver_ads.conversions)   AS ingresos_silver,
  (SELECT COUNT(*) FROM dev.gold_ads.q7_campanas_sin_conversiones)    AS campanas_sin_conv,
  (SELECT COUNT(*) FROM dev.gold_ads.q7_campanas_sin_conversiones q
     WHERE EXISTS (SELECT 1 FROM dev.gold_ads.conversions_atribuidas v WHERE v.campaign_id = q.campaign_id)) AS violaciones_regla_q7,
  (SELECT SUM(impresiones) FROM dev.gold_ads.q8_paises_mejor_conversion) AS imp_q8,
  (SELECT COUNT(*) FROM dev.silver_ads.impressions)                    AS imp_silver,
  (SELECT SUM(conversiones) FROM dev.gold_ads.q8_paises_mejor_conversion) AS conv_q8,
  (SELECT ROUND(SUM(gasto_total_usd), 4) FROM dev.gold_ads.q8_paises_mejor_conversion) AS gasto_q8,
  (SELECT ROUND(SUM(ingresos_usd), 2) FROM dev.gold_ads.q8_paises_mejor_conversion) AS ingresos_q8;

gasto_q6,gasto_q4,conv_q6,conv_silver,ingresos_q6,ingresos_silver,campanas_sin_conv,violaciones_regla_q7,imp_q8,imp_silver,conv_q8,gasto_q8,ingresos_q8
6576.2261,6576.2261,256,256,5829.8,5829.8,1903,0,160000,160000,256,6576.2264,5829.8


## Respuestas de negocio (números reales de la última corrida)

**6. ¿Qué industrias recuperan más por cada dólar invertido (ROAS)?** Solo **Retail (ROAS 1,52)** gana más de lo que gasta: $1.819,68 de ingresos por $1.196,30 invertidos. Las demás están por debajo de 1: FinTech 0,82, Gaming 0,82, SaaS 0,73 y **Travel es la peor (0,58)**: cada dólar invertido devuelve 58 centavos. Lectura de negocio: si hubiese que recortar presupuesto, Travel es la primera candidata. El **ROAS global es 0,89** (se recupera el 89% de lo invertido): Retail está **0,63 puntos ARRIBA** del promedio y Travel **0,31 DEBAJO** — la brecha de cada industria contra el promedio se ve en el gráfico.

**7. ¿Qué campañas gastan sin convertir?** **1.903 de las 2.000 campañas** (el 95%) no tienen ni una conversión. Antes de asustarse: con solo 256 conversiones en total, matemáticamente casi ninguna campaña puede tener una. El valor real es el **ranking por gasto**: la peor es "Enable Global Web Services" (Display, Retail) con $15,27 quemados y ni una conversión. Los gastos individuales son chicos ($10-15), o sea que no hay campañas problemáticas puntuales: el gasto sin retorno está **repartido parejo**. El resumen por canal lo confirma: el gasto quemado total es **$6.084,46 — el 92,5% de TODO el gasto del período** — repartido casi exacto entre los 4 canales (~25% cada uno, Search apenas arriba con 25,5%). Solo ~$492 se gastaron en campañas que sí convirtieron.

**8. ¿Qué países convierten mejor?** El top es **HN (0,99%), LS (0,87%), TL (0,87%), CN (0,84%)** — todos con tasas entre 0,5% y 1% sobre ~800 impresiones. Las tasas son casi parejas entre países, y no es casualidad: el generador asigna el país al azar, así que el país no discrimina el éxito. En datos reales, esta tabla tendría mucho más spread. La economía por país agrega otra capa: el CPA más barato está en **GR ($3,68) y CV ($3,69)**, mientras que HN (el país con MÁS conversiones, 8) tiene CPA $4,87 — convertir más veces no es lo mismo que convertir más barato.

**Validación:** el gasto total de Q6 ($6.576,23) es idéntico al de Q4 por canal; ingresos y conversiones cierran contra silver ($5.829,80 / 256); ninguna campaña de la lista Q7 tiene conversiones (0 violaciones); y el umbral de 500 impresiones no descartó ningún país (los 195 quedaron). El gasto y los ingresos por país también cierran contra el total ($6.576,23 / $5.829,80 — una diferencia de $0,0003 por redondear a 4 decimales por fila, no un error).